## 7. CSV Dosyası Sorgulama (CSV Agent)
* Amaç:  Veri üzerinde doğal dil sorgusu yapmayı öğretmek.
* Senaryo: Kullanıcı bir CSV dosyası yükler ve "en çok satan ürün nedir?" gibi sorular sorar.
* Kazandırılan: Veri analizi + LLM entegrasyonu.
* Kullanılan Bileşenler: create_csv_agent, PandasAgent

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               pandas transformers accelerate sentencepiece


In [ ]:

# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
import pandas as pd
from langchain_community.llms import HuggingFacePipeline
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents import create_pandas_dataframe_agent

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ============================================================
# 3) MODEL: Açık kaynak Qwen
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) CSV YÜKLEME
# ============================================================
# Kaggle'a CSV dosyanızı yükleyin, path aşağıdaki gibi olmalı:
CSV_PATH = '/kaggle/input/retail-data-set/file_out2.csv'  # burayı kendi dosyanıza göre değiştirin
df = pd.read_csv(CSV_PATH)

print("✅ Veri kümesi ilk 5 satır:")
display(df.head())

# ============================================================
# 5) CSV AGENT OLUŞTURMA
# ============================================================
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# ============================================================
# 6) DOĞAL DİL SORGULAR
# ============================================================
questions = [
    "En çok satış yapılan ürün hangisidir?",
    "Toplam gelir ne kadar?",
    "En düşük memnuniyet puanı kaç ve hangi müşteri verdi?",
    "2023 yılında kaç adet sipariş verilmiş?",
    "Fiyatı 100 TL'den fazla olan ürünlerin ortalama adedi kaç?"
]

for i, q in enumerate(questions, 1):
    print(f"\n--- Soru {i}: {q} ---")
    try:
        answer = agent.run(q)
        print(f"Cevap: {answer}")
    except Exception as e:
        print(f"Hata: {e}")
